In [ ]:
import pandas as pd
import numpy as np

from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import make_pipeline

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import spearmanr, pearsonr

import optuna

import shap
import joblib

import warnings
np.random.seed(0) 
warnings.filterwarnings("ignore")


## Feature engineering

In [ ]:
def variance_threshold(X, threshold=0):
    
    selector = VarianceThreshold(threshold)

    feature_names = X.columns
    
    X_new = selector.fit_transform(X)
    
    selected_features_mask = selector.get_support()

    selected_features = feature_names[selected_features_mask]
    
    X_new_df = pd.DataFrame(X_new, columns=selected_features, index=X.index)
    
    return X_new_df

def correlation_threshold(X, y, threshold=0.1):

    correlations = X.corrwith(y)
    
    selected_features_mask = abs(correlations) >= threshold
    
    X_selected = X.loc[:, selected_features_mask]

    selected_correlations = correlations[selected_features_mask]
    
    return X_selected

def combined_feature_selection(data, variance_threshold_val=0.0, correlation_threshold_val=0.1):

    X = data.iloc[:, :-1]
    y = data.iloc[:, -1]
    
    # Step 1: Variance Thresholding
    X_variance_selected = variance_threshold(X, threshold=variance_threshold_val)
    
    # Step 2: Correlation Thresholding
    X_final = correlation_threshold(X_variance_selected, y, threshold=correlation_threshold_val)
    
    final_data = pd.concat([X_final, y], axis=1)
    
    return final_data

### Example
# ID column is not included in feature selection, and the last column is the target variable.
data = pd.read_csv("")
id_col = data.iloc[:, [0]] # Assuming the first column is an ID
data_noID = data.iloc[:, 1:] # Assuming the first column is an ID and should be excluded from feature selection
data_new = combined_feature_selection(data_noID, variance_threshold_val=0.0, correlation_threshold_val=0.1)
data_new = pd.concat([id_col, data_new], axis=1) # Add the ID column back to the final dataset
data_new.to_csv("", index=False)

## Hyperparameter tuning

In [ ]:
### LGBM
def objective_lgbm(trial, X, y, fold=5, state=42):

    param_grid = {
        "n_estimators": trial.suggest_int('n_estimators', 50, 651, step=50),
        "max_depth": trial.suggest_int("max_depth", 1, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.0005, 0.1, step=0.0005),
        "num_leaves": trial.suggest_int("num_leaves", 2, 2**6, step=2),
        "subsample": trial.suggest_float("subsample", 0.2, 1, step=0.1),
        "subsample_freq": trial.suggest_int("subsample_freq", 2, 7, step=1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 1, step=0.1),
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.2, 1, step=0.1),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.1, 1, step=0.1),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 1, step=0.1),
        "random_state": trial.suggest_int("random_state", 0, 500),
    }

    model = LGBMRegressor(**param_grid)

    ### Use CV to evaluate the model
    cv = KFold(n_splits=fold, shuffle=True, random_state=state)
    cv_scores_r2 = []
    cv_scores_mae = []
    cv_scores_mse = []
    cv_scores_pearson = []
    cv_scores_spearman = []
    for idx, (train_idx, test_idx) in enumerate(cv.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        model.fit(X_train, y_train)
      
        preds = model.predict(X_test)
        cv_scores_r2.append(r2_score(y_test, preds))
        cv_scores_mae.append(mean_absolute_error(y_test, preds))
        cv_scores_mse.append(mean_squared_error(y_test, preds))
        cv_scores_pearson.append(pearsonr(y_test, preds)[0])
        cv_scores_spearman.append(spearmanr(y_test, preds)[0])

    return np.mean(cv_scores_mse)

# nested cross-validation
def nested_cv_lgbm(X, y, fold=5, state=42, trials=100):
    outer_cv = KFold(n_splits=fold, shuffle=True, random_state=state)

    outer_cv_r2 = []
    outer_cv_mae = []
    outer_cv_mse = []
    outer_cv_pearson = []
    outer_cv_spearman = []
    best_params = []

    for train_idx, val_idx in outer_cv.split(X, y):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Use Optuna to optimize hyperparameters
        study = optuna.create_study(direction="minimize",sampler=optuna.samplers.TPESampler())
        study.optimize(lambda trial: objective_lgbm(trial, X_train, y_train, fold, state), n_trials=trials)

        # Train the model with the best hyperparameters
        best_trial = study.best_trial
        best_mdl = LGBMRegressor(**best_trial.params).fit(X_train, y_train)

        val_preds = best_mdl.predict(X_val)
        outer_cv_r2.append(r2_score(y_val, val_preds))
        outer_cv_mae.append(mean_absolute_error(y_val, val_preds))
        outer_cv_mse.append(mean_squared_error(y_val, val_preds))
        outer_cv_pearson.append(pearsonr(y_val, val_preds)[0])
        outer_cv_spearman.append(spearmanr(y_val, val_preds)[0])
        best_params.append(best_trial.params)

    return outer_cv_r2, outer_cv_mae, outer_cv_mse, outer_cv_pearson, outer_cv_spearman, best_params

### Example
data = pd.read_csv("")

mean_r2, mean_mae, mean_mse, mean_pearson, mean_spearman, best_params = nested_cv_lgbm(data.iloc[:,1:-1].values, data.iloc[:,-1].values, trials=100)

task_df = pd.DataFrame()
task_df['r2'] = mean_r2
task_df['mae'] = mean_mae
task_df['mse'] = mean_mse
task_df['pearson'] = mean_pearson
task_df['spearman'] = mean_spearman
task_df['r2_mean'] = np.mean(mean_r2)
task_df['mae_mean'] = np.mean(mean_mae)
task_df['mse_mean'] = np.mean(mean_mse)
task_df['pearson_mean'] = np.nanmean(mean_pearson)
task_df['spearman_mean'] = np.nanmean(mean_spearman)
task_df['r2_std'] = np.std(mean_r2)
task_df['mae_std'] = np.std(mean_mae)
task_df['mse_std'] = np.std(mean_mse)
task_df['pearson_std'] = np.nanstd(mean_pearson)
task_df['spearman_std'] = np.nanstd(mean_spearman)
task_df['best_params'] = best_params
task_df.to_csv("", index=False)

In [ ]:
### RF
def objective_rf(trial, X, y, fold=5, state=42):
    
    param_grid = {
            'n_estimators': trial.suggest_int('n_estimators', 50,651,50),
            #'max_depth': trial.suggest_categorical('max_depth', [None, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 10, 1),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
            'max_features': trial.suggest_float('max_features', 0.2, 1, step=0.1), 
            'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 15, 40, 5), 
            'random_state': trial.suggest_int('random_state', 0, 500),
            }

    model = RandomForestRegressor(**param_grid)

    ### Use CV to evaluate the model
    cv = KFold(n_splits=fold, shuffle=True, random_state=state)
    cv_scores_r2 = []
    cv_scores_mae = []
    cv_scores_mse = []
    cv_scores_pearson = []
    cv_scores_spearman = []
    for idx, (train_idx, test_idx) in enumerate(cv.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        model.fit(X_train, y_train)
      
        preds = model.predict(X_test)
        cv_scores_r2.append(r2_score(y_test, preds))
        cv_scores_mae.append(mean_absolute_error(y_test, preds))
        cv_scores_mse.append(mean_squared_error(y_test, preds))
        cv_scores_pearson.append(pearsonr(y_test, preds)[0])
        cv_scores_spearman.append(spearmanr(y_test, preds)[0])

    return np.mean(cv_scores_mse)

# nested CV
def nested_cv_rf(X, y, fold=5, state=42, trials=100):
    outer_cv = KFold(n_splits=fold, shuffle=True, random_state=state)

    outer_cv_r2 = []
    outer_cv_mae = []
    outer_cv_mse = []
    outer_cv_pearson = []
    outer_cv_spearman = []
    best_params = []

    for train_idx, val_idx in outer_cv.split(X, y):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        study = optuna.create_study(direction="minimize",sampler=optuna.samplers.TPESampler())
        study.optimize(lambda trial: objective_rf(trial, X_train, y_train, fold, state), n_trials=trials)

        # Train the model with the best hyperparameters
        best_trial = study.best_trial
        best_mdl = RandomForestRegressor(**best_trial.params).fit(X_train, y_train)

        # Evaluate the model on the validation set
        val_preds = best_mdl.predict(X_val)
        outer_cv_r2.append(r2_score(y_val, val_preds))
        outer_cv_mae.append(mean_absolute_error(y_val, val_preds))
        outer_cv_mse.append(mean_squared_error(y_val, val_preds))
        outer_cv_pearson.append(pearsonr(y_val, val_preds)[0])
        outer_cv_spearman.append(spearmanr(y_val, val_preds)[0])
        best_params.append(best_trial.params)

    return outer_cv_r2, outer_cv_mae, outer_cv_mse, outer_cv_pearson, outer_cv_spearman, best_params

### Example
data = pd.read_csv("")

mean_r2, mean_mae, mean_mse, mean_pearson, mean_spearman, best_params = nested_cv_rf(data.iloc[:,1:-1].values, data.iloc[:,-1].values, trials=100)

task_df = pd.DataFrame()
task_df['r2'] = mean_r2
task_df['mae'] = mean_mae
task_df['mse'] = mean_mse
task_df['pearson'] = mean_pearson
task_df['spearman'] = mean_spearman
task_df['r2_mean'] = np.mean(mean_r2)
task_df['mae_mean'] = np.mean(mean_mae)
task_df['mse_mean'] = np.mean(mean_mse)
task_df['pearson_mean'] = np.nanmean(mean_pearson)
task_df['spearman_mean'] = np.nanmean(mean_spearman)
task_df['r2_std'] = np.std(mean_r2)
task_df['mae_std'] = np.std(mean_mae)
task_df['mse_std'] = np.std(mean_mse)
task_df['pearson_std'] = np.nanstd(mean_pearson)
task_df['spearman_std'] = np.nanstd(mean_spearman)
task_df['best_params'] = best_params
task_df.to_csv("", index=False)

In [ ]:
### SVM
def objective_svm(trial, X, y, fold=5, state=42):

    param_grid = {
        'C': trial.suggest_float('C', 0.1, 100, log=True),  
        'gamma': trial.suggest_float('gamma', 1e-4, 1, log=True), 
        'kernel': trial.suggest_categorical('kernel', ['linear', 'rbf', 'sigmoid']), 
        'epsilon': trial.suggest_float('epsilon', 0.01, 1.0),  
        'tol': trial.suggest_float('tol', 1e-5, 1e-3, log=True),
    }

    # 把 StandardScaler 和 SVR 放进 Pipeline
    model = make_pipeline(
        StandardScaler(),
        SVR(**param_grid)
    )

    ### Use CV to evaluate the model
    cv = KFold(n_splits=fold, shuffle=True, random_state=state)
    cv_scores_r2 = []
    cv_scores_mae = []
    cv_scores_mse = []
    cv_scores_pearson = []
    cv_scores_spearman = []

    for idx, (train_idx, test_idx) in enumerate(cv.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Pipeline 会在当前内层训练折上 fit scaler
        model.fit(X_train, y_train)
      
        preds = model.predict(X_test)

        cv_scores_r2.append(r2_score(y_test, preds))
        cv_scores_mae.append(mean_absolute_error(y_test, preds))
        cv_scores_mse.append(mean_squared_error(y_test, preds))
        cv_scores_pearson.append(pearsonr(y_test, preds)[0])
        cv_scores_spearman.append(spearmanr(y_test, preds)[0])

    return np.mean(cv_scores_mse) 


# nested CV
def nested_cv_svm(X, y, fold=5, state=42, trials=100):
    outer_cv = KFold(n_splits=fold, shuffle=True, random_state=state)

    outer_cv_r2 = []
    outer_cv_mae = []
    outer_cv_mse = []
    outer_cv_pearson = []
    outer_cv_spearman = []
    best_params = []

    for train_idx, val_idx in outer_cv.split(X, y):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # 这里不再提前 StandardScaler
        # 直接把原始 X_train 传入 objective_svm
        study = optuna.create_study(
            direction="minimize",
            sampler=optuna.samplers.TPESampler()
        )

        study.optimize(
            lambda trial: objective_svm(trial, X_train, y_train, fold, state),
            n_trials=trials
        )

        # Train the model with the best hyperparameters
        best_trial = study.best_trial

        # 外层训练时，也用 Pipeline
        best_mdl = make_pipeline(
            StandardScaler(),
            SVR(**best_trial.params)
        )

        best_mdl.fit(X_train, y_train)

        # Evaluate the model on the outer validation set
        # Pipeline 会用外层训练集 fit 出来的 scaler transform X_val
        val_preds = best_mdl.predict(X_val)

        outer_cv_r2.append(r2_score(y_val, val_preds))
        outer_cv_mae.append(mean_absolute_error(y_val, val_preds))
        outer_cv_mse.append(mean_squared_error(y_val, val_preds))
        outer_cv_pearson.append(pearsonr(y_val, val_preds)[0])
        outer_cv_spearman.append(spearmanr(y_val, val_preds)[0])
        best_params.append(best_trial.params)

    return outer_cv_r2, outer_cv_mae, outer_cv_mse, outer_cv_pearson, outer_cv_spearman, best_params

### Example
data = pd.read_csv("")

X = data.iloc[:, 1:-1].values
y = data.iloc[:, -1].values

mean_r2, mean_mae, mean_mse, mean_pearson, mean_spearman, best_params = nested_cv_svm(
    X, y, trials=100
)

task_df = pd.DataFrame()
task_df['r2'] = mean_r2
task_df['mae'] = mean_mae
task_df['mse'] = mean_mse
task_df['pearson'] = mean_pearson
task_df['spearman'] = mean_spearman
task_df['r2_mean'] = np.mean(mean_r2)
task_df['mae_mean'] = np.mean(mean_mae)
task_df['mse_mean'] = np.mean(mean_mse)
task_df['pearson_mean'] = np.nanmean(mean_pearson)
task_df['spearman_mean'] = np.nanmean(mean_spearman)
task_df['r2_std'] = np.std(mean_r2)
task_df['mae_std'] = np.std(mean_mae)
task_df['mse_std'] = np.std(mean_mse)
task_df['pearson_std'] = np.nanstd(mean_pearson)
task_df['spearman_std'] = np.nanstd(mean_spearman)
task_df['best_params'] = best_params
task_df.to_csv("", index=False)

## SHAP

In [ ]:
### RF & LGBM
best_mdl = joblib.load("")
data  = pd.read_csv("")

shap.initjs()
explainer = shap.TreeExplainer(best_mdl)
shap_values = explainer.shap_values(data.iloc[:, 1:-1]) # only features, not including target

shap.summary_plot(shap_values, data.iloc[:, 1:-1])